<a href="https://colab.research.google.com/github/THEJoshinator20/ST-554-Project1-Template/blob/main/Task1/Task1_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Alana Pooler
<br>
ST 554 Project 1

# Task 1: Prediction of C6H6(GT)

In this notebook, we will use the Air Quality data set from the UC Irvine Machine Learning Repository. Our goal is to predict C6H6(GT), the hourly average benzene concentration.

To predict this value, we will write two algorithms to help us find the optimal prediction, c, that minimizes the Root Mean Squared Error of the prediction. We will implement a grid search algorithm as well as a gradient descent algorithm.

We test two versions of each algorithm:


*   One that only considers a given y variable, and c is a constant that minimizes the function.
*   One that considers a linear equation of another numeric variable, and c is given by $\beta_0 + \beta_1x_i$, where $\beta_0$ is the intercept and $\beta_1$ is the slope.
    * In this case, we will use PT08.S1(CO), the hourly averaged sensor response, as the predictor variable.

## Load and clean data set

In [ ]:
# install ucimlrepo
!pip install ucimlrepo

In [2]:
# import libraries
import ucimlrepo as uci
import numpy as np
import pandas as pd
from typing import Optional
from sklearn import linear_model

Read in the data set from the UCI library

In [3]:
air_quality = uci.fetch_ucirepo(id=360)
# view data set
air_quality = air_quality.data.features
air_quality.head()

,Date,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH
0,3/10/2004,18:00:00,2.6,1360,150,11.9,1046,166,1056,113,1692,1268,13.6,48.9,0.7578
1,3/10/2004,19:00:00,2.0,1292,112,9.4,955,103,1174,92,1559,972,13.3,47.7,0.7255
2,3/10/2004,20:00:00,2.2,1402,88,9.0,939,131,1140,114,1555,1074,11.9,54.0,0.7502
3,3/10/2004,21:00:00,2.2,1376,80,9.2,948,172,1092,122,1584,1203,11.0,60.0,0.7867
4,3/10/2004,22:00:00,1.6,1272,51,6.5,836,131,1205,116,1490,1110,11.2,59.6,0.7888


The variables 'C6H6(GT)' and 'CO(GT)' contains missing values, which are coded as '-200'. We will remove these observations for better prediction accuracy.

In [4]:
air_quality_clean = air_quality[(air_quality['C6H6(GT)'] != -200) & (air_quality['CO(GT)'] != -200)]

Since we will mainly be using C6H6(GT) as y and PT08.S1(CO) as x, we will rename these variables to make them easier to reference.

In [5]:
air_quality_clean = air_quality_clean.rename(columns = {'C6H6(GT)': 'y', 'PT08.S1(CO)': 'x'})

 ## Root Mean Squared Error Calculation

 For each algorithm, we will measure prediction quality using Root Mean Squared Error (RMSE). RMSE is given by:

 $\sqrt{\frac{1}{n} \sum_{i=1}^n (y_i - \hat{y_i})^2}$

 For the algorithms using just y, $\hat{y_i}$ is given by c. For algorithms using x and y, $\hat{y_i}$ is given by $c_i = \beta_0 + \beta_1 x_i$.

 Our goal is to find the value of c that minimizes RMSE.

 We will write a generalized function to calculate RMSE that can be used in all four algorithms.

In [6]:
def rmse(
    y: pd.Series,
    x: Optional[pd.Series] = None,
    c: Optional[float] = None,
    b0: Optional[float] = None,
    b1: Optional[float] = None
) -> float:
  """
  Calculates Root Mean Squared Error (RMSE) using either just a y variable, or x and y variables.
  """
  # check if x, b0, and b1 have been given - calculate RMSE using these values if so
  if x is not None and b0 is not None and b1 is not None:
    return np.sqrt(np.mean((y - b0 - b1 * x) ** 2))

  # otherwise, calculate RMSE using just y and c
  else:
    return np.sqrt(np.mean((y - c) ** 2))

## Grid Search Algorithm: Just y

First, we will implement a grid search to find the optimal value of c based off the data set.

This function will create a grid of values to test, starting at the 1st quartile of the given y variable and ending at the 3rd quartile. RMSE will be calculated for each c value in this grid, and the c that results in the smallest RMSE will be returned.

In [7]:
def find_optimal_c(y: pd.Series) -> float:
  """
  Function to find the optimal value of c for a given column.
  """
  # find first and third quartile
  q1, q3 = np.percentile(y, [25, 75])

  # create grid using quartiles
  grid = np.linspace(q1, q3, 100)

  # calculate RMSE for each value in grid
  rmse_values = [rmse(y, c = c) for c in grid]

  # output optimal value of c
  return grid[np.argmin(rmse_values)]

Now, let's test the function on C6H6(GT)

According to calculus, the optimal value of c should be the mean of y. The mean of C6H6(GT) is 10.2757 and the algorithm gave 10.2828, so the it did a pretty good job.

In [8]:
c = find_optimal_c(air_quality_clean['y'])
print(f"Optimal c value = {round(c, 4)}")
print(f"Mean value of C6H6(GT) = {round(air_quality_clean['y'].mean(), 4)}")

Optimal c value = 10.2828
Mean value of C6H6(GT) = 10.2757


Next, we will test the function on PT08.S1(CO) to make sure the algorithm generalizes.

Again, the optimal value of c found by the grid search algorithm is very close to the mean value of PT08.S1(CO), so we can conclude that the algorithm generalizes well.

In [9]:
c = find_optimal_c(air_quality_clean['x'])
print(f"Optimal c value = {round(c, 4)}")
print(f"Mean value of PT08.S1(CO) = {round(air_quality_clean['x'].mean(), 4)}")

Optimal c value = 1109.6364
Mean value of PT08.S1(CO) = 1110.5807


## Grid search algorithm: x and y

Next, we will implement another grid search algorithm, but this time using an x and a y variable. In this model, the predictions $c_i$ are given by:

$$c_i = \beta_0 + \beta_1 x_i$$

To do this, we will create a grid of $\beta_0$ and $\beta_1$ values to calculate RMSE for, and the values that result in the smallest RMSE will be returned.

In [10]:
def find_optimal_betas(y: pd.Series, x: pd.Series):
  """
  Finds optimal values of b0 and b1 for a given x and y variable.
  """

  # create b0 and b1 values
  b0_vals, b1_vals = np.arange(-25, -15, 0.1), np.arange(-5, 5, 0.01)

  # create grid of b0 and b1 values
  grid = [(b0, b1) for b0 in b0_vals for b1 in b1_vals]

  # calculate RMSE for each value of b0 and b1
  rmse_values = [rmse(y, x, b0 = b0, b1 = b1) for b0, b1 in grid]

  # find best values of b0 and b1
  best_b0, best_b1 = grid[np.argmin(rmse_values)]

  return best_b0, best_b1


Now we will test the algorithm using PT08.S1(CO) as x and C6H6(GT) as y.

We will fit a simple linear regression model at the end to see how these values compare to the values given by the SLR model.

In [11]:
b0, b1 = find_optimal_betas(air_quality_clean['y'], air_quality_clean['x'])
print(f"Optimal b0 = {round(b0, 4)}, optimal b1 = {round(b1, 4)}")

Optimal b0 = -23.0, optimal b1 = 0.03


Now, use the values found for $\beta_0$ and $\beta_1$ to predict new values of 'C6H6(GT)' for a 'PT08.S1(CO)' of 946, 1075, and 1246.

Predicted value for PT08.S1(CO) = 946: 5.38
<br>
Predicted value for PT08.S1(CO) = 1075: 9.25
<br>
Predicted value for PT08.S1(CO) = 1246: 14.38

In [12]:
# use for loop to calculate values of C6H6(GT) using given values of PT08.S1(CO)
for num in [946, 1075, 1246]:
  pred = b0 + b1 * num
  print(round(pred, 4))

5.38
9.25
14.38


## Gradient Descent Algorithm: Just y

Now we will switch to using a gradient descent algorithm. This is an optimization algorithm that iteratively adjusts the parameters of the prediction model to find the minimum RMSE.

First, we need to write a function that calculates the difference quotient to use in our gradient descent algorithms.

We will generalize this function so that it can be used in the gradient descent algorithm with just y, as well as the gradient descent algorithm with x and y.

In [13]:
def diff_quotient(
    y: pd.Series,
    delta: float,
    param: str,
    x: Optional[pd.Series] = None,
    c: Optional[float] = None,
    b0: Optional[float] = None,
    b1: Optional[float] = None
) -> float:
  """
  Calculate difference quotient to approximate the slope of tangent line using a given y, c, and delta, or a given x, y, b0, b1, and delta.
  """

  # calculte regular RMSE (without delta)
  base_rmse = rmse(y = y, x = x, c = c, b0 = b0, b1 = b1)

  # check which parameter is given and calculate RMSE accordingly
  if param == 'c':
    rmse_delta = rmse(y = y, c = c + delta)

  elif param == 'b0':
    rmse_delta = rmse(y = y, x = x, b0 = b0 + delta, b1 = b1)

  elif param == 'b1':
    rmse_delta = rmse(y = y, x = x, b0 = b0, b1 = b1 + delta)

  # return difference quotient
  return (rmse_delta - base_rmse) / delta

Now we will write a function for the gradient descent algorithm. This algorithm will:
* Start at a given value of c
* Calculate the difference quotient at that value of c
* Update c by moving a small step in the negative direction of the difference quotient
* Check if the absolute value of the difference between the updated c and the previous c is greater than a predefined tolerance value
    * If it is, the loop will stop and the updated value of c will be returned
    * Otherwise, the loop will repeat with the updated value of c as the new starting value

In [14]:
def gradient_descent_y(
    y: pd.Series,
    start_value: int = 0,
    delta: float = 0.001,
    step_size: float = 0.01,
    tolerance: float = 0.0001,
    max_iterations: int = 10000
) -> float:
  """
  Calculate optimal value of c using a gradient descent algorithm.
  """

  # assign starting value to cur_c
  cur_c = start_value

  for i in range(max_iterations):
    # calculate new_c
    new_c = cur_c - (diff_quotient(y, delta, param = 'c', c = cur_c)) * step_size

    # stop loop if abs(new_c - cur_c) < num_tol
    if abs(new_c - cur_c) < tolerance:
      break

    # otherwise, update cur_c and repeat
    cur_c = new_c

  return cur_c

Now let's test the function on C6H6(GT), using 0 as the starting value of c.

The result is very similar to the value of c we got from the grid search algorithm, although the grid search algorithm resulted in a c value that was slightly closer to the mean of C6H6(GT).

In [15]:
c = gradient_descent_y(air_quality_clean['y'], start_value = 0)
print(f"Optimal c value for C6H6(GT) = {round(c, 4)}")

Optimal c value for C6H6(GT) = 10.2009


Next we will test the function on PT08.S1(CO) to make sure the algorithm generalizes, using 1100 as the starting value of c.

This value of c is closer to the mean of PT08.S1(CO) than the c value we got from the grid search algorithm. We can conclude that the algorithm does generalize well.

In [16]:
c = gradient_descent_y(air_quality_clean['x'], start_value = 1100, step_size = 0.1)
print(f"Optimal c value for PT08.S1(CO) = {round(c, 4)}")

Optimal c value for PT08.S1(CO) = 1110.3616


## Gradient Descent Algorithm: Using x and y

This algorithm will be similar to the previous gradient descent algorithm, except it will find the optimal parameters $\beta_0$ and $\beta_1$ instead of just c.



In [17]:
def gradient_descent_xy(
    y: pd.Series,
    x: pd.Series,
    delta_b0: float = 0.005,
    delta_b1: float = 0.005,
    start_b0: int = -20,
    start_b1: int = 0,
    step_size_b0: float =  0.5,
    step_size_b1: float = 0.00005,
    tolerance: float = 0.0001,
    max_iterations: int = 100000
):
  """
  Run a gradient descent algorithm to find optimal b0 and b1 values.
  """

  # get starting values for b0 and b1
  cur_b0, cur_b1 = start_b0, start_b1

  # loop to keep calculating b0 and b1 until max # of iterations is reached
  for i in range(max_iterations):

    # calculate difference quotient for b0
    diff_quotient_b0 = diff_quotient(y, delta_b0, 'b0', x, b0 = cur_b0, b1 = cur_b1)
    # calculate new b0 value
    new_b0 = cur_b0 - diff_quotient_b0 * step_size_b0

    # calculate difference quotient for b1 using new b0 value
    diff_quotient_b1 = diff_quotient(y, delta_b1, 'b1', x, b0 = new_b0, b1 = cur_b1)
    # calculate new b1
    new_b1 = cur_b1 - diff_quotient_b1 * step_size_b1

    # calculate euclidean distance between current b0, b1 and new b0, b1
    distance = np.linalg.norm(np.array([new_b0, new_b1]) - np.array([cur_b0, cur_b1]))

    # stop loop if euclidian distance is less than tolerance
    if distance < tolerance:
      break

    # otherwise, update cur_c and repeat
    cur_b0, cur_b1 = new_b0, new_b1

  return cur_b0, cur_b1

Now, test the function using C6H6(GT) as y and PT08.S1(CO) as x.

The value found for $\beta_0$ is pretty close to the value we found using the grid search algorithm, but the $\beta_1$ value is a bit further off. We will calculate the values using a simple linear regression model below to check the results.

In [18]:
gd_b0, gd_b1 = gradient_descent_xy(y = air_quality_clean['y'], x = air_quality_clean['x'])
gd_b0, gd_b1

(np.float64(-22.30482975822852), np.float64(-0.0014903666811793767))

Let's find the values of $\beta_0$ and $\beta_1$ using a simple linear regression model to compare to the results of the gradient descent algorithm.

The values of $\beta_0$ and $\beta_1$ that we got from the gradient descent algorithm are very close to the values given by the SLR model, and the values given by the grid search algorithm are closer (-23 and 0.03)

In [143]:
reg = linear_model.LinearRegression()
reg.fit(air_quality_clean['x'].values.reshape(-1,1), air_quality_clean['y'])
print(f"Optimal b0 from SLR = {reg.intercept_}")
print(f"Optimal b1 from SLR = {reg.coef_}")

Optimal b0 from SLR = -22.95574738007751
Optimal b1 from SLR = [0.02992262]


Now, we will use the values of $\beta_0$ and $\beta_1$ found by the gradient descent algorithm to predict a new C6H6(GT) for a PT08.S1(CO) of 946, 1075, and 1246.

In [19]:
# use for loop to calculate new values of C6H6(GT) using given values of PT08.S1(CO)
for num in [946, 1075, 1246]:
  pred = gd_b0 + gd_b1 * num
  print(round(pred, 4))

-23.7147
-23.907
-24.1618


Takeaways:

The grid search algorithm performed slightly better than the gradient descent algorithm when using just y as well as when using x and y. The values found for c, $\beta_0$ and $\beta_1$ were slightly closer to the values found using calculus based methods than the gradient descent algorithm was.

The run time for the grid search algorithm is also shorter, especially when using x and y, compared to the run time for the gradient descent algorithm.